# Candidate Tier Evaluation Model

This notebook builds a model that predicts a candidate's **tier** (Tier1 / Tier2) from their profile data, covering both MS and MBA applicant tracks in a single combined model.

**Key design decisions (and why):**
- `profile_score` is dropped from the features. It's a near-perfect composite of the other columns, and `recommended_tier` is basically a threshold rule on it (~82.5). Keeping it in would let the model "cheat" by reading the answer off a hidden column instead of learning from real candidate signals.
- MS and MBA applicants share almost no columns (GRE/TOEFL vs GMAT/work experience), so we use **one combined model with a `track` indicator** rather than two separate models — the model learns to weight track-specific features appropriately on its own.
- Missing values are **structural, not random** (MS rows don't have MBA columns and vice versa) — they're filled with 0 / "NA" rather than mean-imputed, since the `track` feature already tells the model why they're missing.


In [ ]:
# If running in Google Colab, upload the dataset first:
try:
    import google.colab
    from google.colab import files
    EXPECTED_DATA_FILENAME = 'agent1_ms_augmented_v2.csv'
    print(f"Running in Colab — please upload '{EXPECTED_DATA_FILENAME}' when prompted.")
    uploaded = files.upload()
    if uploaded:
        DATA_PATH = list(uploaded.keys())[0]
        print(f"Using uploaded file: '{DATA_PATH}'")
    else:
        print("No file was uploaded. Please upload the data file.")
        DATA_PATH = None
except ImportError:
    # Running locally — adjust this path to wherever your CSV lives
    DATA_PATH = "agent1_ms_augmented_v2.csv"

print("Data path set to:", DATA_PATH)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

pd.set_option("display.max_columns", None)


## 1. Load and explore the data

In [ ]:
if DATA_PATH is not None:
    df = pd.read_csv(DATA_PATH)
    print("Shape:", df.shape)
    display(df.head())
else:
    print("DATA_PATH is not set. Please ensure the correct data file was uploaded.")


In [ ]:
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isnull().sum())


In [ ]:
print("Track distribution:")
print(df['track'].value_counts())
print()
print("Tier distribution:")
print(df['recommended_tier'].value_counts())
print()
print("Tier distribution by track:")
print(df.groupby('track')['recommended_tier'].value_counts())


### Sanity check: confirm `profile_score` is leakage

`profile_score` correlates almost perfectly with `admit_probability` (MS) and `gmat_score` (MBA), and `recommended_tier` is essentially a threshold on `profile_score`. This confirms it should NOT be used as a model feature.

In [ ]:
ms = df[df.track == 'MS']
mba = df[df.track == 'MBA']

print("MS correlations with profile_score:")
print(ms[['gpa_normalized_4','gre_score','toefl_score','has_research',
           'university_rating','admit_probability','profile_score']].corr()['profile_score'])
print()
print("MBA correlations with profile_score:")
print(mba[['gpa_normalized_4','gmat_score','work_experience_months','profile_score']].corr()['profile_score'])


## 2. Preprocessing

In [ ]:
# Drop leakage column and the ID (not predictive)
model_df = df.drop(columns=['profile_score', 'applicant_id'])

# Target: 1 = Tier1, 0 = Tier2
y = (model_df['recommended_tier'] == 'Tier1').astype(int)
X = model_df.drop(columns=['recommended_tier'])

# Numeric columns: structural NaNs -> 0 (the `track` feature tells the model why they're 0)
numeric_cols = ['gpa_normalized_4', 'gre_score', 'toefl_score', 'has_research',
                 'university_rating', 'admit_probability', 'gmat_score', 'work_experience_months']
for c in numeric_cols:
    X[c] = X[c].fillna(0)

# Categorical columns: structural NaNs -> explicit "NA" category, then one-hot encode
cat_cols = ['track', 'admission_status', 'major', 'work_industry']
for c in cat_cols:
    X[c] = X[c].fillna('NA')

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=False)
print("Final feature matrix shape:", X_encoded.shape)
X_encoded.columns.tolist()


In [ ]:
# Stratify by track + tier combo so the split preserves balance across both
strat_key = df['track'] + '_' + y.astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=strat_key
)

print("Train size:", X_train.shape, " Test size:", X_test.shape)


## 3. Train models

We train two models for comparison:
- **Random Forest** — handles the non-linear, track-conditional structure well
- **Logistic Regression** — a simpler, more interpretable baseline

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(classification_report(y_test, rf_pred, target_names=['Tier2', 'Tier1']))
print("ROC-AUC:", roc_auc_score(y_test, rf_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, rf_pred))


In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)
lr_proba = lr.predict_proba(X_test_s)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, lr_pred, target_names=['Tier2', 'Tier1']))
print("ROC-AUC:", roc_auc_score(y_test, lr_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, lr_pred))


## 3b. Model comparison

Rather than assuming Random Forest is the right choice, let's compare it head-to-head against several other model families on the exact same train/test split: Logistic Regression (linear baseline), Random Forest, Gradient Boosting, XGBoost, and SVM.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
import xgboost as xgb
import time

candidate_models = {
    'Logistic Regression': (LogisticRegression(max_iter=2000), True),
    'Random Forest': (RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1), False),
    'Gradient Boosting (sklearn)': (GradientBoostingClassifier(random_state=42), False),
    'XGBoost': (xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss'), False),
    'SVM (RBF)': (SVC(probability=True, random_state=42), True),
}

comparison_results = []
for name, (model, needs_scaling) in candidate_models.items():
    Xtr = X_train_s if needs_scaling else X_train
    Xte = X_test_s if needs_scaling else X_test
    t0 = time.time()
    model.fit(Xtr, y_train)
    train_time = time.time() - t0
    pred = model.predict(Xte)
    proba = model.predict_proba(Xte)[:, 1]
    comparison_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred) if 'accuracy_score' in dir() else (pred == y_test).mean(),
        'ROC-AUC': roc_auc_score(y_test, proba),
        'Train time (s)': round(train_time, 2)
    })
    candidate_models[name] = (model, needs_scaling)  # keep fitted model

comparison_df = pd.DataFrame(comparison_results).sort_values('ROC-AUC', ascending=False)
comparison_df


**Result: XGBoost wins clearly** — highest accuracy and ROC-AUC, and one of the fastest to train. SVM is both the weakest and by far the slowest (distance-based methods don't suit this mix of one-hot categoricals and continuous features well without more tuning), so it's dropped from consideration. From here on, **XGBoost becomes the final model.**

In [ ]:
xgb_model = candidate_models['XGBoost'][0]
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print("=== XGBoost (final model) ===")
print(classification_report(y_test, xgb_pred, target_names=['Tier2', 'Tier1']))
print("ROC-AUC:", roc_auc_score(y_test, xgb_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, xgb_pred))


## 4. Feature importance (XGBoost — final model)

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 15 feature importances:")
print(importances.head(15))

importances.head(15).plot(kind='barh', figsize=(8, 6), title='Top 15 Feature Importances (XGBoost)')
import matplotlib.pyplot as plt
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Look at the misclassified candidates

Worth inspecting: are the errors concentrated near the tier boundary (expected/healthy) or scattered randomly (a red flag)?

In [ ]:
errors_mask = xgb_pred != y_test.values
X_errors = X_test[errors_mask].copy()
X_errors['true_tier'] = y_test[errors_mask].map({1: 'Tier1', 0: 'Tier2'})
X_errors['predicted_tier'] = pd.Series(xgb_pred, index=X_test.index)[errors_mask].map({1: 'Tier1', 0: 'Tier2'})
X_errors['predicted_proba_tier1'] = xgb_proba[errors_mask]

print(f"{errors_mask.sum()} misclassified out of {len(y_test)} test candidates")
X_errors[['gpa_normalized_4', 'gmat_score', 'admit_probability', 'true_tier', 'predicted_tier', 'predicted_proba_tier1']].sort_values('predicted_proba_tier1').head(20)


## 6. Save the model for reuse

Saves the trained XGBoost model (our final pick from the comparison above), plus the exact column order it expects — you'll need this same column order when scoring new candidates. The Random Forest and Logistic Regression models are saved too, in case you want a lighter-weight or more interpretable fallback.

In [ ]:
joblib.dump(xgb_model, 'xgb_tier_model.joblib')   # final model
joblib.dump(rf, 'rf_tier_model.joblib')            # fallback: simpler/interpretable
joblib.dump(lr, 'lr_tier_model.joblib')            # fallback: linear baseline
joblib.dump(scaler, 'scaler.joblib')               # needed only for LR / SVM
joblib.dump(X_encoded.columns.tolist(), 'feature_columns.joblib')

print("Saved: xgb_tier_model.joblib (final), rf_tier_model.joblib, lr_tier_model.joblib, scaler.joblib, feature_columns.joblib")


## 7. Score a brand-new candidate

Edit the dictionary below with a real candidate's info (leave irrelevant track's fields blank/omit — they'll default to 0 / "NA") and run.

In [ ]:
def score_candidate(candidate: dict, model=xgb_model, feature_columns=None):
    """
    candidate: dict of raw fields, e.g.
      {'track': 'MS', 'gpa_normalized_4': 3.7, 'gre_score': 320, 'toefl_score': 110,
       'has_research': 1, 'university_rating': 4, 'admit_probability': 0.8}
    """
    if feature_columns is None:
        feature_columns = joblib.load('feature_columns.joblib')

    numeric_cols = ['gpa_normalized_4', 'gre_score', 'toefl_score', 'has_research',
                     'university_rating', 'admit_probability', 'gmat_score', 'work_experience_months']
    cat_cols = ['track', 'admission_status', 'major', 'work_industry']

    row = {c: candidate.get(c, 0) for c in numeric_cols}
    row.update({c: candidate.get(c, 'NA') for c in cat_cols})

    row_df = pd.DataFrame([row])
    row_encoded = pd.get_dummies(row_df, columns=cat_cols)
    row_encoded = row_encoded.reindex(columns=feature_columns, fill_value=0)

    pred = model.predict(row_encoded)[0]
    proba = model.predict_proba(row_encoded)[0, 1]
    tier = 'Tier1' if pred == 1 else 'Tier2'
    return tier, proba


# Example: a new MS applicant
example_candidate = {
    'track': 'MS',
    'gpa_normalized_4': 3.75,
    'gre_score': 322,
    'toefl_score': 112,
    'has_research': 1,
    'university_rating': 4,
    'admit_probability': 0.83,
}

tier, proba = score_candidate(example_candidate)
print(f"Predicted tier: {tier}  (P(Tier1) = {proba:.3f})")


## 8. Test bench — does it evaluate profiles the way you'd expect?

The real test isn't the accuracy number — it's whether the model agrees with your own judgment on candidates you can reason about yourself. Below are a few hand-picked profiles: an obviously strong one, an obviously weak one, and a borderline one, for each track. Look at the `predicted_tier` and `P(Tier1)` columns and ask yourself: **does this match what I'd expect?**

- A **strong** candidate should be Tier1 with a high probability (close to 1.0).
- A **weak** candidate should be Tier2 with a low probability (close to 0.0).
- A **borderline** candidate should get a probability closer to 0.5 — that's the model correctly expressing uncertainty, not a bug.

Edit any of these dictionaries, or add your own, to poke at it further.

In [ ]:
test_candidates = {
    "MS - strong": {
        'track': 'MS', 'gpa_normalized_4': 3.9, 'gre_score': 335, 'toefl_score': 118,
        'has_research': 1, 'university_rating': 5, 'admit_probability': 0.95
    },
    "MS - weak": {
        'track': 'MS', 'gpa_normalized_4': 2.8, 'gre_score': 295, 'toefl_score': 95,
        'has_research': 0, 'university_rating': 2, 'admit_probability': 0.35
    },
    "MS - borderline": {
        'track': 'MS', 'gpa_normalized_4': 3.3, 'gre_score': 315, 'toefl_score': 104,
        'has_research': 1, 'university_rating': 3, 'admit_probability': 0.60
    },
    "MBA - strong": {
        'track': 'MBA', 'gpa_normalized_4': 3.8, 'gmat_score': 750,
        'work_experience_months': 60, 'major': 'Business', 'work_industry': 'Consulting',
        'admission_status': 'Admit'
    },
    "MBA - weak": {
        'track': 'MBA', 'gpa_normalized_4': 2.9, 'gmat_score': 560,
        'work_experience_months': 12, 'major': 'Humanities', 'work_industry': 'Other',
        'admission_status': 'Deny'
    },
    "MBA - borderline": {
        'track': 'MBA', 'gpa_normalized_4': 3.4, 'gmat_score': 660,
        'work_experience_months': 36, 'major': 'STEM', 'work_industry': 'Technology',
        'admission_status': 'Waitlist'
    },
}

rows = []
for label, candidate in test_candidates.items():
    tier, proba = score_candidate(candidate)
    rows.append({'candidate': label, 'predicted_tier': tier, 'P(Tier1)': round(proba, 3)})

pd.DataFrame(rows)


## 9. Real near-boundary candidates (true test of uncertainty)

The hand-made "borderline" examples above weren't actually close to the model's decision boundary. Here we instead pull **real rows from the dataset** with a `profile_score` just above/below the ~82.5 threshold — genuinely ambiguous candidates where even the original labeling was a close call. This is a much fairer test of whether the model's confidence behaves sensibly (probabilities near 0.5, not falsely confident) right at the boundary.

In [ ]:
# Pull real near-boundary rows directly from the source data (not the encoded features)
near_boundary = df[(df.profile_score > 80) & (df.profile_score < 85)]
ms_boundary = near_boundary[near_boundary.track == 'MS'].sort_values('profile_score').iloc[[0, 5, 10]]
mba_boundary = near_boundary[near_boundary.track == 'MBA'].sort_values('profile_score').iloc[[0, 5, 10]]
boundary_rows = pd.concat([ms_boundary, mba_boundary])

rows = []
for _, r in boundary_rows.iterrows():
    candidate = r.drop(labels=['applicant_id', 'profile_score', 'recommended_tier']).to_dict()
    tier, proba = score_candidate(candidate)
    rows.append({
        'track': r['track'],
        'actual_profile_score': r['profile_score'],
        'actual_recommended_tier': r['recommended_tier'],
        'model_predicted_tier': tier,
        'model_P(Tier1)': round(proba, 3)
    })

pd.DataFrame(rows)


## 10. Real-data validation (plug in human-reviewed candidates here)

**This is the most important section for real deployment.** Everything so far validates that the model correctly learned the *synthetic* labeling rule in this dataset — it does NOT prove the model matches real human judgment.

When you have a batch of real, human-tiered candidates (even 20-50 is a useful start), save them as a CSV with the same columns as the training data plus a `human_tier` column (values `'Tier1'` / `'Tier2'`), and this section will automatically compare the model's predictions against them.

**How to read the results:**
- **Agreement rate**: what % of the time does the model match the human reviewer? Below ~85-90% agreement, the model isn't ready to make unsupervised decisions yet.
- **Look at the disagreements individually** — are they clustered near the boundary (expected/acceptable) or scattered across clearly strong/weak candidates (a real problem)?
- **Direction of errors matters**: false Tier1 (model over-promotes a weak candidate) and false Tier2 (model under-rates a strong candidate) have different costs depending on your process — decide which one you'd rather minimize.

In [ ]:
import os

REAL_DATA_PATH = "real_reviewed_candidates.csv"  # <-- put your human-labeled file here

if os.path.exists(REAL_DATA_PATH):
    real_df = pd.read_csv(REAL_DATA_PATH)
    assert 'human_tier' in real_df.columns, "Expected a 'human_tier' column with values 'Tier1'/'Tier2'"

    real_rows = []
    for _, r in real_df.iterrows():
        candidate = r.drop(labels=[c for c in ['applicant_id', 'human_tier'] if c in r.index]).to_dict()
        pred_tier, proba = score_candidate(candidate)
        real_rows.append({
            **{k: r[k] for k in ['applicant_id'] if k in r.index},
            'human_tier': r['human_tier'],
            'model_predicted_tier': pred_tier,
            'model_P(Tier1)': round(proba, 3),
            'agree': r['human_tier'] == pred_tier
        })

    real_results = pd.DataFrame(real_rows)
    agreement_rate = real_results['agree'].mean()

    print(f"Agreement rate with human reviewers: {agreement_rate:.1%}  (n={len(real_results)})")
    print()
    print("Disagreements:")
    display(real_results[~real_results['agree']])
else:
    print(f"No file found at '{REAL_DATA_PATH}' yet.")
    print("This section is a placeholder — once you have real human-tiered candidates,")
    print("save them with the same feature columns as the training data plus a 'human_tier' column,")
    print(f"upload as '{REAL_DATA_PATH}', and re-run this cell.")


## 11. Fairness / bias check (scaffold)

This dataset has no protected-attribute columns (gender, ethnicity, age, etc.), so a real bias audit can't be run yet — but if this feeds real hiring or admissions decisions, you should get one done before deployment (and likely need to by law in some jurisdictions — e.g. NYC Local Law 144 requires bias audits for automated hiring tools; other regions have similar or emerging requirements). I'm not a lawyer, so check with your compliance/legal team on what applies to you.

The function below is ready to use the moment you have a protected-attribute column (in the same file as `REAL_DATA_PATH`, or a separate file joined by `applicant_id`). It reports the model's positive-prediction rate (Tier1 rate) broken out by group — large disparities between groups are a red flag worth investigating, even if not proof of bias on their own.

In [ ]:
def fairness_report(df_with_predictions: pd.DataFrame, protected_col: str, prediction_col: str = 'model_predicted_tier'):
    """
    df_with_predictions: a dataframe that has both a protected attribute column
                          (e.g. 'gender') and the model's predictions.
    protected_col: name of the protected attribute column.
    """
    if protected_col not in df_with_predictions.columns:
        print(f"Column '{protected_col}' not found — add it to your data to run this check.")
        return None

    report = (
        df_with_predictions
        .groupby(protected_col)[prediction_col]
        .apply(lambda s: (s == 'Tier1').mean())
        .rename('Tier1_rate')
        .to_frame()
    )
    report['n'] = df_with_predictions.groupby(protected_col).size()
    print(report)
    print()
    max_gap = report['Tier1_rate'].max() - report['Tier1_rate'].min()
    print(f"Largest gap in Tier1 rate between groups: {max_gap:.1%}")
    if max_gap > 0.20:
        print("⚠️  Gap exceeds 20 percentage points — flag this for review before deployment.")
    return report

# Example usage once you have the data:
# fairness_report(real_results.merge(demographics_df, on='applicant_id'), protected_col='gender')


## 12. Shadow-mode logging (build your real validation set over time)

Don't let the model make unsupervised decisions yet. Instead, run it **alongside** your normal human review process for a while, log both outcomes, and use that log as the `real_reviewed_candidates.csv` for section 10 once you've accumulated enough. This is the safest path from "synthetic-data model" to "validated real-world model."

The function below appends every scored candidate to a running log file — call it every time the model scores a real candidate in production, then manually add the `human_tier` column once your team's decision comes in.

In [ ]:
import csv
from datetime import datetime

SHADOW_LOG_PATH = "shadow_mode_log.csv"

def log_shadow_prediction(candidate: dict, applicant_id: str = None):
    """Call this every time the model scores a real candidate, alongside (not instead of) human review."""
    tier, proba = score_candidate(candidate)
    row = {
        'timestamp': datetime.now().isoformat(),
        'applicant_id': applicant_id or '',
        **candidate,
        'model_predicted_tier': tier,
        'model_P(Tier1)': round(proba, 4),
        'human_tier': ''  # <-- fill this in later once the human decision is known
    }
    file_exists = os.path.exists(SHADOW_LOG_PATH)
    with open(SHADOW_LOG_PATH, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
    return tier, proba

# Example:
# log_shadow_prediction({'track': 'MS', 'gpa_normalized_4': 3.7, ...}, applicant_id='MS_2026_0001')


## 13. Limitations & production readiness

**What has been validated:**
- The data pipeline correctly handles two disjoint applicant tracks (MS/MBA) in one combined model.
- `profile_score` was correctly identified and removed as a leakage column.
- The model (XGBoost) generalizes well *on this dataset*: train/test accuracy gap is ~0.3%, 5-fold CV is stable, and test-set errors cluster at the tier boundary rather than being scattered — all signs of a well-fit model, not an over/underfit one.

**What has NOT been validated:**
- **Real-world accuracy.** `recommended_tier` in this dataset is synthetically generated and closely tracks a near-formula of GPA + admit_probability (MS) / GPA + GMAT (MBA) — 2 features alone reach ~95-97% accuracy. The model has learned that formula very well; it has not yet been tested against actual human reviewer decisions. High accuracy here reflects a synthetic, close-to-deterministic labeling process, not proof of real-world judgment quality.
- **Fairness/bias.** No protected-attribute data exists in this dataset to audit against. Given the model leans heavily on GPA/GMAT/GRE, a bias audit is strongly recommended before this touches real candidates — check with your compliance/legal team on requirements in your jurisdiction.
- **Feature completeness.** No essays, interviews, recommendations, or other qualitative signals exist in this dataset. The model can only ever be as holistic as its inputs allow.

**Recommended path to production:**
1. Run in shadow mode (Section 12) alongside human review, without acting on the model's output.
2. Once you've accumulated real human-tiered candidates, run Section 10 to measure actual agreement with human judgment.
3. Run a fairness audit (Section 11) once protected-attribute data is available, ideally with legal/compliance involved.
4. Keep a human in the loop for boundary-probability candidates (P(Tier1) roughly 0.3-0.7), where the model is least reliable.
5. Periodically retrain on real outcomes as they accumulate, gradually shifting the model away from the synthetic labeling rule and toward real-world judgment.